# SPARQL Service Health Check

**Project:** Tests  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6, 2026-07-29)

**Purpose:** Verify that a Wikibase SPARQL endpoint is reachable and returning results.  
Edit **cell 2** to point at a different Wikibase instance, then run all cells.

**Tests:**
- Endpoint reachable — `ASK { ?s ?p ?o }` returns `true`
- Triplestore has data — triple count is greater than zero
- Items have labels — at least one labelled entity exists

In [ ]:
# ── Target Wikibase instance ──────────────────────────────────────────────────
# Edit these two values to point at a different wiki.
# They can also be overridden via ../.env

WIKI_NAME  = 'wikibase.kewl.org'                       # human-readable label
SPARQL_URL = 'https://query.kewl.org/sparql'  # SPARQL endpoint URL

# ── Load overrides from .env (stdlib only — no python-dotenv needed) ──────────
import os
from pathlib import Path

_env_file = Path('../.env')
if _env_file.exists():
    with open(_env_file) as _f:
        for _line in _f:
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _, _v = _line.partition('=')
                os.environ.setdefault(_k.strip(), _v.strip().strip('"\''))

WIKI_NAME  = os.getenv('WIKI_NAME',  WIKI_NAME)
SPARQL_URL = os.getenv('SPARQL_URL', SPARQL_URL)

print(f'Target : {WIKI_NAME}')
print(f'SPARQL : {SPARQL_URL}')

In [ ]:
# ── SPARQL helper (stdlib only — no requests needed) ──────────────────────────
import json, urllib.request, urllib.parse

def _sparql(query, timeout=15):
    params = urllib.parse.urlencode({'query': query, 'format': 'json'})
    req = urllib.request.Request(
        f'{SPARQL_URL}?{params}',
        headers={'Accept': 'application/sparql-results+json'},
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        ct = resp.headers.get('Content-Type', '')
        body = resp.read().decode()
    if 'json' not in ct:
        raise ValueError(f'Expected JSON but got {ct!r}. Body: {body[:300]}')
    return json.loads(body)

# ── Test runner ───────────────────────────────────────────────────────────────
_results = []

def _test(label, fn):
    try:
        detail = fn()
        _results.append((label, True))
        print(f'  PASS  {label}' + (f'  ->  {detail}' if detail else ''))
    except Exception as exc:
        _results.append((label, False))
        print(f'  FAIL  {label}  ->  {exc}')

# ── Tests ─────────────────────────────────────────────────────────────────────
_results.clear()
print(f'Health check: {WIKI_NAME}')
print(f'Endpoint    : {SPARQL_URL}')
print('─' * 60)

def t1():
    data = _sparql('ASK { ?s ?p ?o }', timeout=15)
    assert data.get('boolean') is True, f'unexpected response: {data}'
    return 'ASK returned true'
_test('Endpoint reachable', t1)

def t2():
    data = _sparql('SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }', timeout=30)
    n = int(data['results']['bindings'][0]['n']['value'])
    assert n > 0, 'triplestore is empty'
    return f'{n:,} triples'
_test('Triplestore has data', t2)

def t3():
    data = _sparql('SELECT ?item ?label WHERE { ?item rdfs:label ?label } LIMIT 5', timeout=30)
    bindings = data['results']['bindings']
    assert bindings, 'no labelled items found'
    qid = bindings[0]['item']['value'].split('/')[-1]
    lbl = bindings[0]['label']['value']
    return f'{len(bindings)} labelled item(s)  (e.g. {qid}: "{lbl}")'
_test('Items have labels', t3)

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(1 for _, ok in _results if ok)
total  = len(_results)
print('─' * 60)
verdict = 'ALL PASS ✓' if passed == total else f'FAILED {total - passed}/{total} ✗'
print(f'Result: {passed}/{total} passed  |  {verdict}')